# Acquire and Minimize

In [ ]:
# This file opens one .csv file, looks for CAS numbers, downloads a 3d .sdf from PubChem to local filesystem, 
# then creates a miminimized version of .sdf in separate folder.

## Configuration

In [1]:
# Project configuration. Change to match your project.
projectdirname = "/erligand260"
cascsv = "cas.csv"


In [2]:
# Code configurations. Do not change.
from pathlib import Path
notebookdir = str(Path.cwd())

ligandrootdir = notebookdir + "/ligand"
rawdir =  "/raw"
processeddir = "/processed"

rawliganddir = ligandrootdir + projectdirname + rawdir
processedliganddir = ligandrootdir + projectdirname + processeddir
cascsvfile = ligandrootdir + projectdirname + "/" + cascsv

## Optional - Lookup and Download Pubchem data by CAS number

In [ ]:
# This block is optional. This code might be useful in the future as it collects additional fields from PubChem.
# Use CAS numbers to download meta data from PubChem that includes a unique CID identifier. Save that data into a variable.
import pandas as pd
import pubchempy as pcp

outdf = None

def ask_pubchem(casfile):
    # Input CSV must have a column named 'CAS'
    df = pd.read_csv(casfile)
    
    results = []
    
    for cas in df["CAS"].dropna():
        cas = str(cas).strip()
        try:
            compounds = pcp.get_compounds(cas, namespace="name")
            if compounds:
                c = compounds[0]
                results.append({
                    "CAS": cas,
                    "CID": c.cid,
                    "MolecularFormula": c.molecular_formula,
                    "MolecularWeight": c.molecular_weight,
                    "IUPACName": c.iupac_name
                })
            else:
                results.append({
                    "CAS": cas,
                    "CID": None,
                    "MolecularFormula": None,
                    "MolecularWeight": None,
                    "IUPACName": None
                })
        except Exception as e:
            results.append({
                "CAS": cas,
                "CID": None,
                "MolecularFormula": None,
                "MolecularWeight": None,
                "IUPACName": f"ERROR: {e}"
            })

    return (results)

In [ ]:
# Create a new list for results from PubChem function above and call ask_pubchem() function
pubchemresults = []
pubchemresults.extend(ask_pubchem(cascsvfile))
print(pubchemresults)

In [ ]:
outdf = pd.DataFrame(pubchemresults)
# print to screen
print (outdf)
# or save to filesystem
# outdf.to_csv(ligandrootdir + projectdirname + "/temp.csv", index=False)

## Download .sdf files from PubChem based on list of CAS files in .csv file

In [5]:
# Open .csv file and populate CAS list variable
import csv
cas_list = []
with open(cascsvfile, newline="", encoding="utf-8") as f:
    r = csv.reader(f)
    #next(r, None) #Ignore first row.
    cas_list = [row[0] for row in r if row]

print(str(len(cas_list)) + " rows in " + cascsvfile)
print(cas_list)

261 rows in /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/cas.csv
['CAS', '57-63-6', '84-16-2', '56-53-1', '84-17-3', '53-16-7', '72-33-3', '57-91-0', '50-27-1', '50-28-2', '34816-55-2', '474-86-2', '5976-61-4', '53-63-4', '362-05-0', '120-47-8', '72-55-9', '72-43-5', '53-19-0', '59-50-7', '207-13-4', '1570-64-5', '72-54-8', '97-54-1', '85-68-7', '90-43-7', '99-76-3', '92-69-3', '140-10-3', '131-57-7', '115-29-7', '115-86-6', '123-07-9', '84-69-5', '481-30-1', '84-74-2', '571-22-2', '218-01-9', '131-56-6', '143-74-8', '58-89-9', '60-57-1', '50-02-2', '81-90-3', '80-09-1', '2050-68-2', '521-18-6', '1806-26-4', '1852-53-5', '15372-34-6', '13037-86-0', '94-25-7', '1085-12-7', '599-64-4', '57-88-5', '94-26-8', '98-54-4', '58-22-0', '28034-99-3', '50-22-6', '94-18-8', '243-17-4', '53-45-2', '121-33-5', '80-46-6', '97-53-0', '50-29-3', '80-05-7', '92-04-6', '580-51-8', '6554-98-9', '103-16-2', '94-13-3', '58-72-0', '83-46-5', '611-99-4', '99-71-8', '126-00-1', '789-02-6

In [19]:
from pathlib import Path
sdf_count = len(list(Path(rawliganddir).glob("*.sdf")))
print("Filecount for folder " + rawliganddir + ": " + str(sdf_count))

Filecount for folder /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw: 248


In [12]:
# Block of code to download the 3d versions of .sdf files from PubChem.
# This code downloads .sdf files if they are not already present and places them into the 'raw' folder.
# This code often fails to download one or a few files for unknown reasons. This code will print that failure message to the screen.
# The human can then use the examples in a later block to download these missed files individually.
# Following list format expected:
# cas_list = ["CAS","120-47-8","72-55-9","72-43-5"]


import pubchempy as pcp
import time
from pathlib import Path

counter = 1

faillist = []

# Remove header text if necessary
if cas_list and cas_list[0] == 'CAS':
    cas_list.pop(0)
    

for cas in cas_list:
    outsdf = rawliganddir + "/cas-" + cas + ".sdf"
    outsdf_path = Path(outsdf)
    if outsdf_path.is_file():
        print("Line #" + str(counter) + ": File " + outsdf + " already present.")
    else:
        try:
            matches = pcp.get_compounds(cas, namespace="name")
            if matches:
              cid = matches[0].cid
              print ("Line #" + str(counter) + ": Download and write " + outsdf)
              pcp.download("SDF", outsdf, cid, namespace='cid', overwrite=True, record_type='3d')
              time.sleep(4.0)
            else:
              print("Line #" + str(counter) + ": No match found for " + cas + ". Sorry. :(")
              faillist.append( "Line #" + str(counter) + ": No match found for " + cas + ". Sorry. :(")
        except Exception as e:
            msg = str(e)
            print("  Line #" + str(counter) + ": Error for " + cas + ": " + msg)
            faillist.append("Line #" + str(counter) + ": Error for " + cas + ": " + msg)
    counter+=1

sdf_count = len(list(Path(rawliganddir).glob("*.sdf")))
print("Filecount: " + str(sdf_count))

Line #1: File /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw/cas-57-63-6.sdf already present.
Line #2: File /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw/cas-84-16-2.sdf already present.
Line #3: File /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw/cas-56-53-1.sdf already present.
Line #4: File /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw/cas-84-17-3.sdf already present.
Line #5: File /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw/cas-53-16-7.sdf already present.
Line #6: File /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw/cas-72-33-3.sdf already present.
Line #7: File /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw/cas-57-91-0.sdf already present.
Line #8: File /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/erligand260/raw/cas-50-27-1.sdf already present.
Line #9: File /home/dwaine/octoberproject/gninaw

TypeError: can only concatenate str (not "int") to str

In [14]:
# Print out failed CAS downloads.
for fail in faillist:
    print(fail)

Line #20: No match found for 207-13-4. Sorry. :(
Line #159: No match found for 84852-15-3. Sorry. :(
Line #169: Error for 57-11-4: PubChem HTTP Error 404 PUGREST.NotFound
Line #175: Error for 612-82-8: PubChem HTTP Error 404 PUGREST.NotFound
Line #178: Error for 960-71-4: PubChem HTTP Error 404 PUGREST.NotFound
Line #189: Error for 28553-12-0: PubChem HTTP Error 404 PUGREST.NotFound
Line #190: Error for 915-67-3: PubChem HTTP Error 404 PUGREST.NotFound
Line #203: No match found for COULD NOT FIND. Sorry. :(
Line #211: Error for 117-81-7: PubChem HTTP Error 404 PUGREST.NotFound
Line #240: Error for 54760-75-7: PubChem HTTP Error 404 PUGREST.NotFound
Line #251: Error for 103-23-1: PubChem HTTP Error 404 PUGREST.NotFound
Line #253: Error for 117-84-0: PubChem HTTP Error 404 PUGREST.NotFound


## Download missed files individually

In [ ]:
# Now we need to look up and manually download the missed file.
# We need to find the chemical, probably by it's name listed in the original .xls file, in the pubchem website.
# We are looking for a page similar to the one below.
# https://pubchem.ncbi.nlm.nih.gov/compound/75575
# We can look at the page to identify the CID number.
# That CID allows me to WGET the file. Here is a sample that writes the file to the correct location.
# Here is the generic wget command example
# wget -O 207-13-4_3d.sdf https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/75575/SDF?record_type=3d

# Here is the python code that calls wget and directs wget to save in the appropriate folder.
#saveto = rawliganddir + "/cas-207-13-4.sdf"
#!wget -O "{saveto}" https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/75575/SDF?record_type=3d

In [ ]:
# Modify this block and execute as necessary
#saveto = rawliganddir + "/cas-207-13-4.sdf"
#!wget -O "{saveto}" https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/CID/75575/SDF?record_type=3d

## Minimize the SDF files

In [ ]:
# Minimize the .sdf files using obabel command line tool. 
# This code based on  mfaber_workflow/workflow_notebook/Ligand_Prep_Trial.ipynb
# These minimized files will be saved into a separate 'processed' directory.

import os

for entry in os.scandir(rawliganddir):
    if entry.is_file() and entry.name.endswith(".sdf"):
        
        stem = os.path.splitext(entry.name)[0]
        out = processedliganddir + "/" + stem + "_min.sdf"
        print("Convert " + entry.path + " to " + out)
        !obabel "{entry.path}" -O "{out}" -p --minimize --ff mmff94
